In [2]:
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

# ----------------------------
# 1) Load
# ----------------------------
INPUT_PATH = Path("./tvpvar_preprocessed/tvpvar_raw_level_merged.csv")

df = pd.read_csv(INPUT_PATH)
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

# 사용할 변수 직접 지정
num_cols = [
    "SOLVPN",
    "COPPER",
    "GOLD",
    "SILVER",
    "DXY",
    "UST10Y",
    "VIX",
    "OIL",
    "GAS",
]

# 실제 존재하는 컬럼만 사용
missing_cols = [c for c in num_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing columns in input file: {missing_cols}")

print("Using columns:", num_cols)

# ----------------------------
# 2) 표준화
# ----------------------------
df_std = df[["Date"] + num_cols].copy()

for col in num_cols:
    s = pd.to_numeric(df_std[col], errors="coerce")
    mean = s.mean()
    std = s.std()

    if std == 0 or pd.isna(std):
        raise ValueError(f"Cannot standardize column with zero or NaN std: {col}")

    df_std[col] = (s - mean) / std

# ----------------------------
# 3) Plotly 인터랙티브 그래프
# ----------------------------
fig = go.Figure()

for col in num_cols:
    fig.add_trace(
        go.Scatter(
            x=df_std["Date"],
            y=df_std[col],
            mode="lines",
            name=col
        )
    )

fig.update_layout(
    title="Interactive Raw Level Overlay - Standardized",
    xaxis_title="Date",
    yaxis_title="Z-score",
    height=650,
    hovermode="x unified",
    template="plotly_white",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="left",
        x=0
    )
)

fig.show()

Using columns: ['SOLVPN', 'COPPER', 'GOLD', 'SILVER', 'DXY', 'UST10Y', 'VIX', 'OIL', 'GAS']
